In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Load dataset and handle zero values
url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
df = pd.read_csv(url)

cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols] = df[cols].replace(0, np.nan)
df[cols] = df[cols].fillna(df[cols].median())

X, y = df.drop('Outcome', axis=1), df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Bootstrap function and train 15 decision trees
def bootstrap_sample(X, y):
    idx = np.random.choice(len(X), len(X), replace=True)
    return X.iloc[idx], y.iloc[idx]

np.random.seed(42)
models = []

for i in range(15):
    X_boot, y_boot = bootstrap_sample(X_train, y_train)
    tree = DecisionTreeClassifier(random_state=i)
    tree.fit(X_boot, y_boot)
    models.append(tree)

# 3. Majority voting
def majority_vote(models, X_test):
    predictions = np.array([m.predict(X_test) for m in models])
    return np.apply_along_axis(
        lambda x: np.bincount(x.astype(int)).argmax(),
        axis=0, arr=predictions
    )

y_pred = majority_vote(models, X_test)

# 4. Single Decision Tree
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
single_pred = single_tree.predict(X_test)

# 5. Compare accuracy and confusion matrices
for name, pred in [
    ("Single Decision Tree", single_pred),
    ("Custom Bagging Classifier", y_pred)
]:
    print("\n", name)
    print("Accuracy:", round(accuracy_score(y_test, pred) * 100, 2), "%")
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))


 Single Decision Tree
Accuracy: 68.18 %
Confusion Matrix:
 [[79 21]
 [28 26]]

 Custom Bagging Classifier
Accuracy: 72.73 %
Confusion Matrix:
 [[81 19]
 [23 31]]
